In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
koleksi_dokumen = {
    1: "Perancangan Jaringan VLAN (Virtual Local Area Network) di SMKN 40 Jakarta dengan Menggunakan Metode NDLC (Network Development Life Cycle)",
    2: "Pengembangan Aplikasi Media Pembelajaran Berbasis Android pada Pelajaran Teknik Pengolahan Audio Video di SMK Negeri 7 Jakarta",
    3: "Pengembangan Web Service Sistem Informasi Skripsi untuk Program Studi Teknik Informatika dan Komputer Universitas Negeri Jakarta Menggunakan Metode Scrum",
    4: "Topic Modelling Dokumen Skripsi Program Studi Pendidikan Teknik Informatika dan Komputer Universitas Negeri Jakarta Menggunakan Metode Latent Dirichlet Allocation",
    5: "Pengembangan Sistem Informasi Administrasi Skripsi Smart Management Berbasis Web Program Studi Pendidikan Informatika dan Komputer Universitas Negeri Jakarta",
    6: "Perbandingan Model Prediksi Kebangkrutan Menggunakan Support Vector Machine dan Artificial Neural Network",
    7: "Perbandingan Algoritma K-Nearest Neighbor, Naïve Bayes Classifier, dan Support Vector Machine dalam Klasifikasi Judul Karya Akhir Mahasiswa Program Studi PTIK UNJ",
    8: "Pengembangan Media Pembelajaran Interaktif pada Mata Pelajaran Jaringan Dasar untuk Kelas XI SMK Tunas Teknologi Bekasi",
    9: "Pengembangan Modul Pembelajaran Informatika Terintegrasi Augmented Reality (AR) di SMA Pertiwi 1 Padang",
    10: "Perancangan Sistem Informasi Pengajuan Judul Skripsi Berbasis Web Mahasiswa PTIK Universitas Bung Hatta Padang"
}

In [16]:
dokumen = list(koleksi_dokumen.values())

In [17]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(dokumen)

In [18]:
query = [
    "AI",
    "Jaringan",
    "IoT"
]

In [19]:
ground_truth = {
    "AI": {6, 7},
    "Jaringan": {1, 8},
    "IoT": set()
}

In [20]:
def precision_at_3(ranking, relevant):
    top3 = ranking[:3]
    relevan = sum(1 for doc in top3 if doc in relevant)
    return relevan / 3

In [21]:
def recall_score(ranking, relevant):
    if len(relevant) == 0:
        return 0

    ditemukan = sum(1 for doc in ranking if doc in relevant)
    return ditemukan / len(relevant)

In [22]:
def f1_score(precision, recall):
    if precision + recall == 0:
        return 0

    return 2 * precision * recall / (precision + recall)

In [23]:
def average_precision(ranking, relevant):
    if len(relevant) == 0:
        return 0

    jumlah_relevan = 0
    total_precision = 0

    for posisi, doc in enumerate(ranking, start=1):
        if doc in relevant:
            jumlah_relevan += 1
            precision = jumlah_relevan / posisi
            total_precision += precision

    return total_precision / len(relevant)

In [24]:
hasil = {}

for q in query:
    query_vector = vectorizer.transform([q])

    similarity = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    sorted_indices = similarity.argsort()[::-1]

    ranking = [
        list(koleksi_dokumen.keys())[i]
        for i in sorted_indices
    ]

    relevant = ground_truth[q]

    p3 = precision_at_3(ranking, relevant)
    r = recall_score(ranking, relevant)
    f1 = f1_score(p3, r)
    ap = average_precision(ranking, relevant)

    hasil[q] = {
        "ranking": ranking,
        "scores": similarity,
        "Precision@3": p3,
        "Recall": r,
        "F1": f1,
        "AP": ap
    }

In [25]:
map_score = sum(
    hasil[q]["AP"] for q in query
) / len(query)

In [26]:
for q in query:
    print("\n" + "=" * 70)
    print("QUERY:", q)
    print("=" * 70)

    print("Ground Truth:", ground_truth[q])

    print("\nRanking Dokumen:")

    for rank, doc_id in enumerate(
        hasil[q]["ranking"],
        start=1
    ):
        score = hasil[q]["scores"][doc_id - 1]

        print(
            f"Rank {rank}: Dokumen {doc_id} "
            f"(Score: {score:.4f})"
        )

    print(
        "\nPrecision@3:",
        f"{hasil[q]['Precision@3']:.4f}"
    )

    print(
        "Recall:",
        f"{hasil[q]['Recall']:.4f}"
    )

    print(
        "F1:",
        f"{hasil[q]['F1']:.4f}"
    )

    print(
        "AP:",
        f"{hasil[q]['AP']:.4f}"
    )


QUERY: AI
Ground Truth: {6, 7}

Ranking Dokumen:
Rank 1: Dokumen 10 (Score: 0.0000)
Rank 2: Dokumen 9 (Score: 0.0000)
Rank 3: Dokumen 8 (Score: 0.0000)
Rank 4: Dokumen 7 (Score: 0.0000)
Rank 5: Dokumen 6 (Score: 0.0000)
Rank 6: Dokumen 5 (Score: 0.0000)
Rank 7: Dokumen 4 (Score: 0.0000)
Rank 8: Dokumen 3 (Score: 0.0000)
Rank 9: Dokumen 2 (Score: 0.0000)
Rank 10: Dokumen 1 (Score: 0.0000)

Precision@3: 0.0000
Recall: 1.0000
F1: 0.0000
AP: 0.3250

QUERY: Jaringan
Ground Truth: {8, 1}

Ranking Dokumen:
Rank 1: Dokumen 8 (Score: 0.2336)
Rank 2: Dokumen 1 (Score: 0.2048)
Rank 3: Dokumen 9 (Score: 0.0000)
Rank 4: Dokumen 10 (Score: 0.0000)
Rank 5: Dokumen 6 (Score: 0.0000)
Rank 6: Dokumen 7 (Score: 0.0000)
Rank 7: Dokumen 5 (Score: 0.0000)
Rank 8: Dokumen 4 (Score: 0.0000)
Rank 9: Dokumen 3 (Score: 0.0000)
Rank 10: Dokumen 2 (Score: 0.0000)

Precision@3: 0.6667
Recall: 1.0000
F1: 0.8000
AP: 1.0000

QUERY: IoT
Ground Truth: set()

Ranking Dokumen:
Rank 1: Dokumen 10 (Score: 0.0000)
Rank 2: D

In [ ]:
print("\n" + "=" * 70)
print("MEAN AVERAGE PRECISION (MAP)")
print("=" * 70)
print(f"MAP: {map_score:.4f}")


MEAN AVERAGE PRECISION (MAP)
MAP: 0.4417
